# ASTER (Small) — Attentional Scene Text Recognizer with Flexible Rectification

In [ ]:
import mathimport torchimport torch.nn as nnimport torch.nn.functional as Ffrom torch.utils.data import Dataset, DataLoader

In [ ]:
class Config:    img_height = 32    img_width = 100    n_channels = 1    num_fiducial = 8    d_model = 128    encoder_hidden = 128    attn_dim = 128    vocab_size = 60    max_text_len = 25    pad_id = 0    bos_id = 1    eos_id = 2cfg = Config()device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
class GridGenerator(nn.Module):    def __init__(self, num_fiducial, rectified_size):        super().__init__()        self.eps = 1e-6        self.num_fiducial = num_fiducial        self.rectified_height, self.rectified_width = rectified_size        C = self._build_C(num_fiducial)        P = self._build_P(self.rectified_width, self.rectified_height)        inv_delta_C = self._build_inv_delta_C(num_fiducial, C)        P_hat = self._build_P_hat(num_fiducial, C, P)        self.register_buffer("inv_delta_C", inv_delta_C)        self.register_buffer("P_hat", P_hat)    def _build_C(self, F):        ctrl_pts_x = torch.linspace(-1.0, 1.0, F // 2)        ctrl_pts_y_top = -1 * torch.ones(F // 2)        ctrl_pts_y_bottom = torch.ones(F // 2)        ctrl_pts_top = torch.stack([ctrl_pts_x, ctrl_pts_y_top], dim=1)        ctrl_pts_bottom = torch.stack([ctrl_pts_x, ctrl_pts_y_bottom], dim=1)        C = torch.cat([ctrl_pts_top, ctrl_pts_bottom], dim=0)        return C    def _build_inv_delta_C(self, F, C):        hat_C = torch.zeros(F, F)        for i in range(F):            for j in range(F):                r = torch.norm(C[i] - C[j])                hat_C[i, j] = r        hat_C = (hat_C ** 2) * torch.log(hat_C ** 2 + self.eps)        delta_C = torch.zeros(F + 3, F + 3)        delta_C[:F, :F] = hat_C        delta_C[:F, F] = 1        delta_C[:F, F + 1:] = C        delta_C[F, :F] = 1        delta_C[F + 1:, :F] = C.transpose(0, 1)        inv_delta_C = torch.inverse(delta_C)        return inv_delta_C    def _build_P(self, width, height):        grid_x = (torch.arange(-width, width, 2).float() + 1.0) / width        grid_y = (torch.arange(-height, height, 2).float() + 1.0) / height        P = torch.stack(torch.meshgrid(grid_x, grid_y, indexing="ij"), dim=2)        return P.reshape(-1, 2)    def _build_P_hat(self, F, C, P):        n = P.size(0)        P_tile = P.unsqueeze(1).repeat(1, F, 1)        C_tile = C.unsqueeze(0)        P_diff = P_tile - C_tile        rbf_norm = torch.norm(P_diff, dim=2)        rbf = (rbf_norm ** 2) * torch.log(rbf_norm + self.eps)        P_hat = torch.cat([torch.ones(n, 1), P, rbf], dim=1)        return P_hat    def forward(self, batch_C_prime):        batch_size = batch_C_prime.size(0)        batch_inv_delta_C = self.inv_delta_C.unsqueeze(0).repeat(batch_size, 1, 1)        batch_P_hat = self.P_hat.unsqueeze(0).repeat(batch_size, 1, 1)        zeros = torch.zeros(batch_size, 3, 2, device=batch_C_prime.device)        batch_C_prime_ext = torch.cat([batch_C_prime, zeros], dim=1)        batch_T = torch.bmm(batch_inv_delta_C, batch_C_prime_ext)        batch_P_prime = torch.bmm(batch_P_hat, batch_T)        return batch_P_prime

In [ ]:
class LocalizationNetwork(nn.Module):    def __init__(self, num_fiducial, n_channels):        super().__init__()        self.num_fiducial = num_fiducial        self.conv = nn.Sequential(            nn.Conv2d(n_channels, 32, 3, 1, 1),            nn.BatchNorm2d(32),            nn.ReLU(inplace=True),            nn.MaxPool2d(2, 2),            nn.Conv2d(32, 64, 3, 1, 1),            nn.BatchNorm2d(64),            nn.ReLU(inplace=True),            nn.MaxPool2d(2, 2),            nn.Conv2d(64, 128, 3, 1, 1),            nn.BatchNorm2d(128),            nn.ReLU(inplace=True),            nn.AdaptiveAvgPool2d(1),        )        self.fc1 = nn.Linear(128, 64)        self.fc2 = nn.Linear(64, num_fiducial * 2)        ctrl_pts_x = torch.linspace(-1.0, 1.0, num_fiducial // 2)        ctrl_pts_y_top = -1 * torch.ones(num_fiducial // 2)        ctrl_pts_y_bottom = torch.ones(num_fiducial // 2)        ctrl_pts_top = torch.stack([ctrl_pts_x, ctrl_pts_y_top], dim=1)        ctrl_pts_bottom = torch.stack([ctrl_pts_x, ctrl_pts_y_bottom], dim=1)        initial_bias = torch.cat([ctrl_pts_top, ctrl_pts_bottom], dim=0).reshape(-1)        self.fc2.weight.data.fill_(0.0)        self.fc2.bias.data.copy_(initial_bias)    def forward(self, x):        features = self.conv(x).view(x.size(0), -1)        features = F.relu(self.fc1(features))        batch_C_prime = self.fc2(features).view(-1, self.num_fiducial, 2)        return batch_C_prime

In [ ]:
class TPSRectification(nn.Module):    def __init__(self, cfg):        super().__init__()        self.rectified_size = (cfg.img_height, cfg.img_width)        self.localization = LocalizationNetwork(cfg.num_fiducial, cfg.n_channels)        self.grid_generator = GridGenerator(cfg.num_fiducial, self.rectified_size)    def forward(self, x):        batch_C_prime = self.localization(x)        build_P_prime = self.grid_generator(batch_C_prime)        grid = build_P_prime.reshape(x.size(0), self.rectified_size[0], self.rectified_size[1], 2)        rectified = F.grid_sample(x, grid, padding_mode="border", align_corners=True)        return rectified

In [ ]:
class CNNFeatureExtractor(nn.Module):    def __init__(self, cfg):        super().__init__()        self.layers = nn.Sequential(            nn.Conv2d(cfg.n_channels, 64, 3, 1, 1),            nn.ReLU(inplace=True),            nn.MaxPool2d(2, 2),            nn.Conv2d(64, 128, 3, 1, 1),            nn.ReLU(inplace=True),            nn.MaxPool2d(2, 2),            nn.Conv2d(128, 256, 3, 1, 1),            nn.BatchNorm2d(256),            nn.ReLU(inplace=True),            nn.MaxPool2d((2, 1), (2, 1)),            nn.Conv2d(256, cfg.d_model, 3, 1, 1),            nn.BatchNorm2d(cfg.d_model),            nn.ReLU(inplace=True),            nn.AdaptiveAvgPool2d((1, None)),        )    def forward(self, x):        features = self.layers(x)        features = features.squeeze(2)        return features.permute(0, 2, 1)

In [ ]:
class BiLSTMEncoder(nn.Module):    def __init__(self, cfg):        super().__init__()        self.rnn = nn.LSTM(cfg.d_model, cfg.encoder_hidden, bidirectional=True, batch_first=True)    def forward(self, x):        outputs, _ = self.rnn(x)        return outputs

In [ ]:
class BahdanauAttention(nn.Module):    def __init__(self, encoder_dim, decoder_dim, attn_dim):        super().__init__()        self.encoder_proj = nn.Linear(encoder_dim, attn_dim)        self.decoder_proj = nn.Linear(decoder_dim, attn_dim)        self.energy = nn.Linear(attn_dim, 1)    def forward(self, decoder_hidden, encoder_outputs):        encoder_proj = self.encoder_proj(encoder_outputs)        decoder_proj = self.decoder_proj(decoder_hidden).unsqueeze(1)        scores = self.energy(torch.tanh(encoder_proj + decoder_proj)).squeeze(-1)        weights = F.softmax(scores, dim=-1)        context = torch.bmm(weights.unsqueeze(1), encoder_outputs).squeeze(1)        return context, weights

In [ ]:
class AttentionDecoder(nn.Module):    def __init__(self, cfg, encoder_dim):        super().__init__()        self.embedding = nn.Embedding(cfg.vocab_size, cfg.d_model, padding_idx=cfg.pad_id)        self.attention = BahdanauAttention(encoder_dim, cfg.encoder_hidden, cfg.attn_dim)        self.rnn_cell = nn.LSTMCell(cfg.d_model + encoder_dim, cfg.encoder_hidden)        self.output_layer = nn.Linear(cfg.encoder_hidden, cfg.vocab_size)        self.encoder_hidden = cfg.encoder_hidden        self.bos_id = cfg.bos_id    def forward(self, encoder_outputs, target_tokens=None, max_len=25):        batch_size = encoder_outputs.size(0)        device = encoder_outputs.device        h = torch.zeros(batch_size, self.encoder_hidden, device=device)        c = torch.zeros(batch_size, self.encoder_hidden, device=device)        if target_tokens is not None:            steps = target_tokens.size(1)        else:            steps = max_len        inputs = torch.full((batch_size,), self.bos_id, dtype=torch.long, device=device)        logits_list = []        predictions = []        for t in range(steps):            embedded = self.embedding(inputs)            context, _ = self.attention(h, encoder_outputs)            rnn_input = torch.cat([embedded, context], dim=1)            h, c = self.rnn_cell(rnn_input, (h, c))            logits = self.output_layer(h)            logits_list.append(logits)            predicted = logits.argmax(dim=1)            predictions.append(predicted)            if target_tokens is not None:                inputs = target_tokens[:, t]            else:                inputs = predicted        return torch.stack(logits_list, dim=1), torch.stack(predictions, dim=1)

In [ ]:
class ASTER(nn.Module):    def __init__(self, cfg):        super().__init__()        self.rectification = TPSRectification(cfg)        self.feature_extractor = CNNFeatureExtractor(cfg)        self.encoder = BiLSTMEncoder(cfg)        self.decoder = AttentionDecoder(cfg, encoder_dim=cfg.encoder_hidden * 2)    def forward(self, images, target_tokens=None, max_len=25):        rectified = self.rectification(images)        features = self.feature_extractor(rectified)        encoder_outputs = self.encoder(features)        logits, predictions = self.decoder(encoder_outputs, target_tokens, max_len)        return logits, predictions

In [ ]:
class ToySceneTextDataset(Dataset):    def __init__(self, n_samples, cfg):        self.n_samples = n_samples        self.cfg = cfg    def __len__(self):        return self.n_samples    def __getitem__(self, idx):        image = torch.randn(self.cfg.n_channels, self.cfg.img_height, self.cfg.img_width)        text_len = torch.randint(3, self.cfg.max_text_len - 2, (1,)).item()        tokens = torch.randint(3, self.cfg.vocab_size, (text_len,))        tokens = torch.cat([            torch.tensor([self.cfg.bos_id]),            tokens,            torch.tensor([self.cfg.eos_id])        ])        return image, tokensdef collate_fn(batch):    images, tokens = zip(*batch)    image_batch = torch.stack(images)    max_len = max(t.size(0) for t in tokens)    token_batch = torch.zeros(len(batch), max_len, dtype=torch.long)    for i, t in enumerate(tokens):        token_batch[i, :t.size(0)] = t    return image_batch, token_batch

In [ ]:
dataset = ToySceneTextDataset(n_samples=64, cfg=cfg)loader = DataLoader(dataset, batch_size=8, shuffle=True, collate_fn=collate_fn)model = ASTER(cfg).to(device)optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
def train_epoch(model, loader, optimizer):    model.train()    total_loss = 0.0    for images, tokens in loader:        images = images.to(device)        tokens = tokens.to(device)        decoder_input = tokens[:, :-1]        target = tokens[:, 1:]        logits, _ = model(images, target_tokens=decoder_input, max_len=decoder_input.size(1))        loss = F.cross_entropy(            logits.reshape(-1, cfg.vocab_size), target.reshape(-1), ignore_index=cfg.pad_id        )        optimizer.zero_grad()        loss.backward()        optimizer.step()        total_loss += loss.item()    return total_loss / len(loader)

In [ ]:
for epoch in range(3):    avg_loss = train_epoch(model, loader, optimizer)    print(f"epoch {epoch + 1} loss {avg_loss:.4f}")

In [ ]:
model.eval()with torch.no_grad():    sample_images = torch.randn(2, cfg.n_channels, cfg.img_height, cfg.img_width).to(device)    _, predictions = model(sample_images, target_tokens=None, max_len=cfg.max_text_len)    print(predictions)